In [2]:
import pandas as pd
import numpy as np

# Load merged data
df = pd.read_csv('../data/processed/merged_100k.csv', encoding='utf-8-sig')

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nBasic info:")
print(df.info())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_36296\2897949948.py:5: DtypeWarning: Columns (16,19,20,21,23,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/merged_100k.csv', encoding='utf-8-sig')


Shape: (100000, 27)

Columns: ['id_comment', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate', 'id_product', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category']

Missing values:
title                    49082
body                        10
recommendation_status    14003
advantages               89382
disadvantages            94459
seller_title              4483
seller_code               4483
true_to_size_rate        99550
id_product               87309
title_fa                 87309
Rate                     87309
Rate_cnt                 87309
Category1                87309
Category2                87344
Brand                    87309
Price                    87309
Seller                   87309
Is_Fake                  87309
min_price_last_month     87309
sub_

In [3]:
import pandas as pd
import numpy as np
import re

# Load data
df = pd.read_csv('../data/processed/merged_100k.csv', 
                 encoding='utf-8-sig',
                 low_memory=False)

print(f"Starting with {len(df):,} records\n")

# ===== Text Features =====
print("Creating text features...")

# Combine text fields
df['full_text'] = (
    df['title'].fillna('') + ' ' + 
    df['body'].fillna('') + ' ' + 
    df['advantages'].fillna('') + ' ' + 
    df['disadvantages'].fillna('')
).str.strip()

# Text lengths
df['text_length'] = df['full_text'].str.len()
df['body_length'] = df['body'].fillna('').str.len()
df['title_length'] = df['title'].fillna('').str.len()

# Word counts
df['word_count'] = df['full_text'].str.split().str.len()

# Has advantages/disadvantages
df['has_advantages'] = df['advantages'].notna().astype(int)
df['has_disadvantages'] = df['disadvantages'].notna().astype(int)

# Sentiment proxy (rate-based)
df['sentiment_label'] = pd.cut(df['rate'], 
                                bins=[0, 2, 3, 5], 
                                labels=['negative', 'neutral', 'positive'])

print(f"Text features created.")
print(f"Sentiment distribution:\n{df['sentiment_label'].value_counts()}\n")

# Save checkpoint
df.to_csv('../data/processed/features_step1.csv', 
          index=False, 
          encoding='utf-8-sig')
print("Saved: data/processed/features_step1.csv")


Starting with 100,000 records

Creating text features...
Text features created.
Sentiment distribution:
sentiment_label
positive    67009
neutral     15166
negative     7046
Name: count, dtype: int64

Saved: data/processed/features_step1.csv


In [4]:
import pandas as pd
import numpy as np

# Load step 1
df = pd.read_csv('../data/processed/features_step1.csv', 
                 encoding='utf-8-sig',
                 low_memory=False)

print(f"Loaded {len(df):,} records\n")

# ===== Product Features =====
print("Creating product features...")

# Price change percentage
df['price_change_pct'] = np.where(
    df['min_price_last_month'].notna() & (df['min_price_last_month'] > 0),
    ((df['Price'] - df['min_price_last_month']) / df['min_price_last_month']) * 100,
    np.nan
)

# Has product info
df['has_product_info'] = df['id_product'].notna().astype(int)

# Product popularity (based on Rate_cnt)
df['product_popularity'] = pd.cut(df['Rate_cnt'], 
                                   bins=[0, 10, 50, 200, np.inf],
                                   labels=['low', 'medium', 'high', 'very_high'])

print(f"Product features created.")

# ===== Seller Features =====
print("Creating seller features...")

# Seller rating aggregation (group by seller_code)
seller_stats = df[df['seller_code'].notna()].groupby('seller_code').agg({
    'rate': ['mean', 'count', 'std'],
    'likes': 'sum',
    'dislikes': 'sum'
}).reset_index()

seller_stats.columns = ['seller_code', 'seller_avg_rate', 'seller_comment_count', 
                        'seller_rate_std', 'seller_total_likes', 'seller_total_dislikes']

# Merge back
df = df.merge(seller_stats, on='seller_code', how='left')

# Seller like ratio
df['seller_like_ratio'] = np.where(
    (df['seller_total_likes'] + df['seller_total_dislikes']) > 0,
    df['seller_total_likes'] / (df['seller_total_likes'] + df['seller_total_dislikes']),
    np.nan
)

print(f"Seller features created.")
print(f"\nSample seller stats:")
print(seller_stats.head())

# Save checkpoint
df.to_csv('../data/processed/features_step2.csv', 
          index=False, 
          encoding='utf-8-sig')
print("\nSaved: data/processed/features_step2.csv")


Loaded 100,000 records

Creating product features...
Product features created.
Creating seller features...
Seller features created.

Sample seller stats:
  seller_code  seller_avg_rate  seller_comment_count  seller_rate_std  \
0       3MKUM         2.185443                    79         2.259753   
1       5A52N         3.698269                  8991         1.528035   
2       5A549         4.000000                     2         0.000000   
3       5A553         5.000000                     1              NaN   
4       5A55D         3.414286                     7         1.332202   

   seller_total_likes  seller_total_dislikes  
0                  20                      2  
1                2680                    458  
2                   4                      3  
3                   0                      0  
4                   5                      6  

Saved: data/processed/features_step2.csv


In [10]:
print(df.columns.tolist())
df.head(100)

['id_comment', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate', 'id_product', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category', 'full_text', 'text_length', 'body_length', 'title_length', 'word_count', 'has_advantages', 'has_disadvantages', 'sentiment_label', 'price_change_pct', 'has_product_info', 'product_popularity', 'seller_avg_rate', 'seller_comment_count', 'seller_rate_std', 'seller_total_likes', 'seller_total_dislikes', 'seller_like_ratio']


,id_comment,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,...,sentiment_label,price_change_pct,has_product_info,product_popularity,seller_avg_rate,seller_comment_count,seller_rate_std,seller_total_likes,seller_total_dislikes,seller_like_ratio
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.00,not_recommended,True,252058,NaN,NaN,...,negative,NaN,0,NaN,3.698269,8991.0,1.528035,2680.0,458.0,0.854047
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.00,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],...,NaN,NaN,0,NaN,3.698269,8991.0,1.528035,2680.0,458.0,0.854047
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.00,recommended,True,3331597,NaN,NaN,...,NaN,NaN,0,NaN,3.638070,1031.0,1.583852,144.0,30.0,0.827586
3,18628562,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,28 اسفند 1399,0.00,recommended,True,3331329,NaN,NaN,...,NaN,NaN,0,NaN,3.753949,785.0,1.424670,71.0,45.0,0.612069
4,53301258,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,12 شهریور 1402,3.00,recommended,True,3255700,NaN,NaN,...,neutral,NaN,0,NaN,3.376190,105.0,1.536465,9.0,0.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,43741983,عالی عالی,خیلی عالی بود، واقعا لاک رو زود پاک میکنه و اص...,10 دی 1401,4.00,NaN,True,2401696,NaN,NaN,...,positive,NaN,0,NaN,4.300885,113.0,0.924710,6.0,1.0,0.857143
96,38708977,اکلیل ناخن,عالیه همون رنگی که خواسته بودم ارسال شد سالم ب...,11 شهریور 1401,5.00,recommended,True,4918138,NaN,NaN,...,positive,NaN,0,NaN,3.578049,123.0,1.538670,42.0,2.0,0.954545
97,46220222,برس مو,برس جنس نرم و خوبی داره ۳۰و ۸۰۰ خریدم بنظرم خوبه,8 اسفند 1401,5.00,recommended,True,2345968,"['زیبا', 'قیمت مناسب', 'جنس قابل قبول']",NaN,...,positive,NaN,0,NaN,3.930636,173.0,1.255610,12.0,3.0,0.800000
98,21124641,خوبه,با توجه به قیمت پایینش خیلی خوبه,2 خرداد 1400,3.25,recommended,True,953687,NaN,NaN,...,positive,NaN,0,NaN,3.920455,22.0,1.437976,1.0,0.0,1.000000
